# Analisis dan Peramalan Inflasi Kota Mataram Menggunakan ARIMA

Notebook ini bertujuan untuk melakukan pemodelan dan peramalan inflasi Kota Mataram menggunakan metode ARIMA. Tahapan analisis meliputi pembacaan data, eksplorasi data, uji stasioneritas, pembagian data training dan testing, evaluasi model menggunakan rolling forecast, serta peramalan periode mendatang.


## 1. Import Library

Tahap pertama adalah memanggil seluruh library yang diperlukan dalam proses analisis. Library yang digunakan meliputi Pandas untuk pengolahan data, NumPy untuk komputasi numerik, Plotly untuk visualisasi interaktif, Statsmodels untuk pemodelan ARIMA, dan Scikit-Learn untuk evaluasi model.


In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_error, mean_squared_error


## 2. Membaca Data

Data inflasi dibaca dari file Excel yang tersimpan pada komputer lokal. Dataset terdiri atas periode pengamatan bulanan dan nilai inflasi yang akan digunakan dalam proses pemodelan.


In [ ]:
file_path = r"D:\inflasi kota mataram.xlsx"
df = pd.read_excel(file_path)
df.head()


## 3. Pemeriksaan Struktur Data

Tahap ini dilakukan untuk memastikan bahwa data telah terbaca dengan benar, termasuk nama variabel, tipe data, dan jumlah observasi yang tersedia.


In [ ]:
print(df.columns)
df.info()


## 4. Preprocessing Data

Kolom waktu dikonversi ke format datetime agar dapat dikenali sebagai data runtun waktu. Selanjutnya data diurutkan berdasarkan periode pengamatan sehingga urutan observasi sesuai dengan kronologi waktu.


In [ ]:
df['Bulan'] = pd.to_datetime(df['Bulan'])
df = df.sort_values('Bulan').reset_index(drop=True)
df.head()


## 5. Membentuk Variabel Time Series

Variabel inflasi dipisahkan dari dataset utama untuk digunakan sebagai objek yang akan dimodelkan menggunakan ARIMA.


In [ ]:
y = df['Inflasi (%)']
y.head()


## 6. Visualisasi Data

Visualisasi dilakukan untuk melihat pola pergerakan inflasi dari waktu ke waktu. Grafik ini membantu mengidentifikasi tren, fluktuasi, dan karakteristik umum data sebelum dilakukan pemodelan.


In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=df['Bulan'], y=y, mode='lines', name='Inflasi'))
fig.update_layout(title='Inflasi Kota Mataram',
                  xaxis_title='Periode',
                  yaxis_title='Inflasi (%)')
fig.show()


## 7. Statistik Deskriptif

Statistik deskriptif digunakan untuk memberikan gambaran umum mengenai karakteristik data inflasi, seperti nilai rata-rata, standar deviasi, nilai minimum, dan nilai maksimum.


In [ ]:
y.describe()

## 8. Uji Stasioneritas Menggunakan Augmented Dickey-Fuller (ADF)

Model ARIMA mensyaratkan data yang digunakan bersifat stasioner. Oleh karena itu dilakukan pengujian menggunakan metode Augmented Dickey-Fuller (ADF).


In [ ]:
adf = adfuller(y)
print("ADF Statistic :", round(adf[0],4))
print("p-value       :", round(adf[1],6))


## 9. Menentukan Nilai Differencing (d)

Jika data belum stasioner, maka dilakukan differencing untuk menghilangkan tren dan membuat rata-rata data menjadi konstan sepanjang waktu.


In [ ]:
if adf[1] < 0.05:
    d = 0
else:
    d = 1
print("Nilai d =", d)


## 10. Pembagian Data Training dan Testing

Data dibagi menjadi data training sebesar 80% dan data testing sebesar 20% untuk mengevaluasi kemampuan prediksi model.


In [ ]:
train_size = int(len(y)*0.8)
train = y[:train_size]
test = y[train_size:]
test_dates = df['Bulan'][train_size:]

print("Training :", len(train))
print("Testing  :", len(test))


## 11. Rolling Forecast ARIMA

Evaluasi model dilakukan menggunakan pendekatan rolling forecast. Pada setiap iterasi model dibangun menggunakan seluruh data historis yang tersedia, kemudian digunakan untuk memprediksi satu periode berikutnya.


In [ ]:
p = 1
q = 1

history = list(train)
predictions = []

for t in range(len(test)):
    model = ARIMA(history, order=(p,d,q))
    model_fit = model.fit()
    yhat = model_fit.forecast()[0]
    predictions.append(yhat)
    history.append(test.iloc[t])


## 12. Evaluasi Akurasi Model

Kinerja model dievaluasi menggunakan MAE, RMSE, dan sMAPE.


In [ ]:
mae = mean_absolute_error(test, predictions)
rmse = np.sqrt(mean_squared_error(test, predictions))

smape = np.mean(
    2*np.abs(test.values - np.array(predictions)) /
    (np.abs(test.values) + np.abs(predictions))
) * 100

print("MAE  :", round(mae,4))
print("RMSE :", round(rmse,4))
print("sMAPE:", round(smape,2), "%")


## 13. Estimasi Model Final

Setelah model dievaluasi, seluruh data digunakan untuk membentuk model akhir yang akan digunakan dalam proses peramalan.


In [ ]:
final_model = ARIMA(y, order=(p,d,q))
final_fit = final_model.fit()
print(final_fit.summary())


## 14. Peramalan Dua Periode Mendatang

Model ARIMA final digunakan untuk menghasilkan prediksi inflasi pada dua periode berikutnya.


In [ ]:
forecast_2 = final_fit.forecast(steps=2)

future_dates = pd.date_range(
    start=df['Bulan'].iloc[-1] + pd.offsets.MonthBegin(1),
    periods=2,
    freq='MS'
)

forecast_df = pd.DataFrame({
    'Bulan': future_dates,
    'Forecast Inflasi': forecast_2.values
})

forecast_df


## 15. Visualisasi Hasil Peramalan

Grafik berikut menampilkan data aktual, hasil rolling forecast pada data testing, dan hasil peramalan untuk periode mendatang.


In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter(x=df['Bulan'], y=y, mode='lines', name='Data Aktual'))
fig.add_trace(go.Scatter(x=test_dates, y=predictions, mode='lines', name='Rolling Forecast'))
fig.add_trace(go.Scatter(x=future_dates, y=forecast_2, mode='lines+markers', name='Forecast'))

fig.update_layout(
    title='Forecast Inflasi Menggunakan ARIMA',
    xaxis_title='Periode',
    yaxis_title='Inflasi (%)',
    template='plotly_white'
)

fig.show()


## 16. Menyimpan Hasil Forecast

Hasil peramalan disimpan dalam format Excel sehingga dapat digunakan untuk analisis lanjutan maupun penyusunan laporan penelitian.


In [ ]:
forecast_df.to_excel('Forecast_ARIMA.xlsx', index=False)
print('Forecast berhasil disimpan.')
